In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from category_encoders import OneHotEncoder, CatBoostEncoder, MEstimateEncoder
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from catboost import CatBoostClassifier, Pool
import os
from sklearn.metrics import accuracy_score,confusion_matrix, ConfusionMatrixDisplay

## Read Source Files

In [ ]:
df_train=pd.read_csv("/kaggle/input/playground-series-s4e2/train.csv")
df_test=pd.read_csv("/kaggle/input/playground-series-s4e2/test.csv")
df_train_org=pd.read_csv("/kaggle/input/obesity-or-cvd-risk-classifyregressorcluster/ObesityDataSet.csv")
sample_sub=pd.read_csv("/kaggle/input/playground-series-s4e2/sample_submission.csv")

In [ ]:
df_train=df_train.drop('id',axis=1)
df_test=df_test.drop('id',axis=1)
df_train=pd.concat([df_train,df_train_org])
df_train=df_train.drop_duplicates()
df_train.head()

# Generic

In [ ]:
targetMap = {'Insufficient_Weight':0,'Normal_Weight':1,
             'Overweight_Level_I':2,'Overweight_Level_II':3, 
             'Obesity_Type_I':4,'Obesity_Type_II':5 ,'Obesity_Type_III':6}
##
df_predictions = pd.DataFrame()
##
num_cols = df_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df_train.select_dtypes(include=['object']).columns.tolist()
cat_cols.remove('NObeyesdad')

## LightGBM

In [ ]:
def getFeatsLgb(df,dft,cat_cols,num_cols):
    df=pd.get_dummies(df,columns=cat_cols)
    sc=StandardScaler()
    sc.fit(dft[num_cols])
    df[num_cols]=sc.transform(df[num_cols])
    return df

In [ ]:
lgb_params = {
    "objective": "multiclass",
    "metric": "multi_logloss",
    "verbosity": -1,
    "boosting_type": "gbdt",
    "random_state": 42,
    "num_class": 7,
    'learning_rate': 0.031,
    'n_estimators': 550,
    'lambda_l1': 0.01,
    'lambda_l2': 0.04,
    'max_depth': 10,
    'colsample_bytree': 0.41,
    'subsample': 0.95,
    'min_child_samples': 26
}

In [ ]:
train_lgb=getFeatsLgb(df_train.copy(),df_train.copy(),cat_cols,num_cols)
test_lgb=getFeatsLgb(df_test.copy(),df_train.copy(),cat_cols,num_cols)
#
X=train_lgb.drop('NObeyesdad',axis=1)
y=train_lgb.NObeyesdad.map(targetMap)

In [ ]:
LGB=LGBMClassifier(**lgb_params)
LGB.fit(X,y)
##
lgb_pred=LGB.predict_proba(test_lgb)
for i,j in targetMap.items():
    df_predictions[f"LGB_{i}"] = lgb_pred[:,j]

## XGBoost

In [ ]:
def getFeatsXgb(df,dft,cat_cols,num_cols):
    
    MEE_encoder = MEstimateEncoder()
    MEE_encoder.fit(dft[cat_cols], df_train.NObeyesdad.map(targetMap))
    df[cat_cols] = MEE_encoder.transform(df[cat_cols])
    sc=StandardScaler()
    sc.fit(dft[num_cols])
    df[num_cols]=sc.transform(df[num_cols])
    
    return df

In [ ]:
xgb_params = {'grow_policy': 'depthwise', 
              'n_estimators': 1000, 
               'learning_rate': 0.05, 'gamma': 0.535, 
               'subsample': 0.706, 'colsample_bytree': 0.38, 
               'max_depth': 23, 'min_child_weight': 21, 
              'reg_lambda': 9.15,
               'reg_alpha': 5.67e-08,
               'booster':'gbtree',
               'objective':'multi:softmax',
               'verbosity':0
              }

In [ ]:
train_xgb=getFeatsXgb(df_train.copy(),df_train.copy(),cat_cols,num_cols)
test_xgb=getFeatsXgb(df_test.copy(),df_train.copy(),cat_cols,num_cols)
#
X=train_xgb.drop('NObeyesdad',axis=1)
y=train_xgb.NObeyesdad.map(targetMap)

In [ ]:
XGB=XGBClassifier(**xgb_params)
XGB.fit(X,y)
##
xgb_pred=XGB.predict_proba(test_xgb)
for i,j in targetMap.items():
    df_predictions[f"XGB_{i}"] = xgb_pred[:,j]

## CATBoost

In [ ]:
def getFeatsCat(df,dft,cat_cols,num_cols):
    sc=StandardScaler()
    sc.fit(dft[num_cols])
    df[num_cols]=sc.transform(df[num_cols])
    return df

In [ ]:
cat_params = {'learning_rate': 0.14, 
              'depth': 5, 
              'l2_leaf_reg': 5.3, 
              'bagging_temperature': 0.6,
              'iterations':1500,
              'random_seed': 42,
              'verbose': 200}

In [ ]:
train_cat=getFeatsCat(df_train.copy(),df_train.copy(),cat_cols,num_cols)
test_cat=getFeatsCat(df_test.copy(),df_train.copy(),cat_cols,num_cols)
#
X=train_cat.drop('NObeyesdad',axis=1)
y=train_cat.NObeyesdad.map(targetMap)
cat_features = np.where(X.dtypes != np.float64)[0]
train_pool = Pool(X, y,cat_features=cat_features)

In [ ]:
CAT = CatBoostClassifier(**cat_params)
CAT.fit(train_pool)

cat_pred=CAT.predict_proba(test_cat)

for i,j in targetMap.items():
    df_predictions[f"CAT_{i}"] = cat_pred[:,j]

## Ensemble and Submission

### Ensemble and Apply Thresholds

## Threshold Credit to : https://www.kaggle.com/code/samlakhmani/easy-92-196-single-model
## Weights Credit to : https://www.kaggle.com/code/ksevta/ps4e2-xgb-lgbm-0-92

In [ ]:
def apply_thresholds(y_proba, thresholds):
    y_pred_labels = np.argmax(y_proba, axis=1)
    for i in range(y_proba.shape[1]):
        y_pred_labels[y_proba[:, i] > thresholds[f'threshold_{i}']] = i

    return y_pred_labels

thresh = {'threshold_0': 0.72, 'threshold_1': 0.62, 
          'threshold_2': 0.29, 'threshold_3': 0.31, 
          'threshold_4': 0.85, 'threshold_5': 0.68, 
          'threshold_6': 0.36}
#

In [ ]:
APPLY_THRESH=False
#
if APPLY_THRESH:
    for i,j in targetMap.items():
        df_predictions[f"{i}"] = (0.72*df_predictions[f"LGB_{i}"]+0.27*df_predictions[f"XGB_{i}"]+0.01*df_predictions[f"CAT_{i}"])
    #
    y_proba=df_predictions[targetMap.keys()].to_numpy()
    preds=apply_thresholds(y_proba, thresh)
    inv_targetMap = {v: k for k, v in targetMap.items()}
    sample_sub['NObeyesdad'] = preds
    sample_sub['NObeyesdad'] = sample_sub['NObeyesdad'].map(inv_targetMap)
else:
    for i,j in targetMap.items():
        df_predictions[f"{i}"] = (5*df_predictions[f"LGB_{i}"]+2*df_predictions[f"XGB_{i}"]+1*df_predictions[f"CAT_{i}"])

    preds = df_predictions[targetMap.keys()].idxmax(axis = 1)

    sample_sub['NObeyesdad'] = preds

### Submission

In [ ]:
sample_sub.to_csv("submission.csv",index=False)
sample_sub